# Praktikum Data Processing Workflow
## Modul: Automated Data Contracts & Assertions (Data Cleaning & Transformation)

**Nama Mahasiswa** : Udi Adhahaska Arta Ananta  
**NRP**            : 3324600052  
**Program Studi**  : D4 Sains Data Terapan  
**Mata Kuliah**    : Data Processing Workflow  

---
### Alur Arsitektur Data Contract Pipeline:
```text
Data Producer (Order Service) 
       │
       ▼
 Incoming Batch (CSV)
       │
       ▼
 [ DATA CONTRACT ]
 ├── Pydantic v2 (Row-level & Business Rule)
 └── Great Expectations (Dataset-level Expectations)
       │
   Quality Gate
   ├── PASS ──► Muat ke Data Warehouse (warehouse_orders.csv)
   └── FAIL ──► Fail-Fast Reject (Data Cacat Ditolak)
```

### Persiapan Environment & Verifikasi Pustaka
Memastikan pustaka Pydantic (v2), Pandas, dan Great Expectations terinstal dan siap digunakan.

In [1]:
import pydantic
import pandas as pd
import great_expectations as gx

print(f"[OK] Pydantic version : {pydantic.__version__}")
print(f"[OK] Pandas version   : {pd.__version__}")
print(f"[OK] Great Exp version: {gx.__version__}")

[OK] Pydantic version : 2.14.0
[OK] Pandas version   : 3.0.3
[OK] Great Exp version: 1.24.0


---
## Percobaan 1: Membuat Dataset Valid dan Invalid
Dataset disiapkan ke dalam direktori `data/` sebagai representasi batch transaksi e-commerce:
- `orders_valid.csv`: 4 baris data yang mematuhi seluruh spesifikasi kontrak.
- `orders_invalid.csv`: 5 baris data yang mengandung beragam pelanggaran (ID negatif, nominal negatif, status asing, dan pelanggaran aturan pembayaran).

In [2]:
df_valid = pd.read_csv('data/orders_valid.csv')
df_invalid = pd.read_csv('data/orders_invalid.csv')

print("=== DATASET VALID (orders_valid.csv) ===")
print(df_valid)

print("\n=== DATASET INVALID (orders_invalid.csv) ===")
print(df_invalid)

=== DATASET VALID (orders_valid.csv) ===
  order_id  customer_id    amount     status     transaction_date         payment_date
0  ORD-001          101  250000.0       PAID  2026-09-20T08:00:00  2026-09-20T08:15:00
1  ORD-002          102  125000.0    PENDING  2026-09-20T08:05:00                  NaN
2  ORD-003          103   75000.0  CANCELLED  2026-09-20T08:10:00                  NaN
3  ORD-004          104  500000.0       PAID  2026-09-20T08:20:00  2026-09-20T08:30:00

=== DATASET INVALID (orders_invalid.csv) ===
  order_id  customer_id    amount     status     transaction_date         payment_date
0  ORD-101          201  250000.0       PAID  2026-09-20T09:00:00  2026-09-20T09:15:00
1  ORD-102         -202  125000.0    PENDING  2026-09-20T09:05:00                  NaN
2  ORD-103          203  -75000.0       PAID  2026-09-20T09:10:00                  NaN
3  ORD-104          204   50000.0    UNKNOWN  2026-09-20T09:15:00                  NaN
4  ORD-105          205  100000.0       PAI

---
## Percobaan 2: Memahami Data Contract & Analisis Pertanyaan Kunci

### Tabel Spesifikasi Kontrak Data:
| Field | Required | Tipe | Constraint / Aturan |
|---|---|---|---|
| `order_id` | Ya | `string` | min_length=1, tidak boleh string kosong |
| `customer_id` | Ya | `integer` | > 0 (harus bilangan positif) |
| `amount` | Ya | `float` | >= 0 (tidak boleh negatif) |
| `status` | Ya | `enum/Literal` | 'PAID', 'PENDING', atau 'CANCELLED' |
| `transaction_date` | Ya | `datetime` | Valid ISO datetime |
| `payment_date` | Tidak | `datetime/None` | Wajib terisi jika `status == 'PAID'` |

### Pembahasan Jawaban Pertanyaan Evaluasi:
1. **Field Identifier**: `order_id` merupakan kunci pengenal unik (*primary key*) setiap pesanan.
2. **Field Wajib Memiliki Nilai**: `order_id`, `customer_id`, `amount`, `status`, dan `transaction_date`.
3. **Field Memiliki Batasan Range**: `customer_id` (> 0) dan `amount` (>= 0.0).
4. **Field Memiliki Allowed Values**: `status` dibatasi hanya pada set `{'PAID', 'PENDING', 'CANCELLED'}`.
5. **Aturan Business Rule**: Hubungan kondisional antar-kolom di mana pesanan dengan `status == 'PAID'` diwajibkan memiliki `payment_date` yang tidak kosong.
6. **Aksi Saat Satu Record Gagal**: Dicatat dalam error log / dead-letter queue beserta detail pelanggaran skema untuk inspeksi.
7. **Aksi Saat Satu Batch Gagal**: Dalam arsitektur *Fail-Fast*, seluruh batch ditolak (*rejected*) dan pemuatan ke Data Warehouse dihentikan untuk menjaga integritas data.

---
## Percobaan 3: Implementasi Data Contract dengan Pydantic v2
Menerapkan `OrderContract` dengan konfigurasi ketat: `strict=True` (menolak konversi implisit yang berbahaya), `extra='forbid'` (menolak penambahan kolom liar), dan decorator `@model_validator(mode='after')` untuk mengecek business rule pembayaran.

In [3]:
from datetime import datetime
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field, ValidationError, model_validator

class OrderContract(BaseModel):
    model_config = ConfigDict(
        strict=True,
        extra="forbid",
    )

    order_id: str = Field(min_length=1)
    customer_id: int = Field(gt=0)
    amount: float = Field(ge=0)
    status: Literal["PAID", "PENDING", "CANCELLED"]
    transaction_date: datetime
    payment_date: datetime | None = None

    @model_validator(mode="after")
    def paid_requires_payment_date(self):
        if self.status == "PAID" and self.payment_date is None:
            raise ValueError("payment_date wajib diisi jika status=PAID")
        return self

print("[OK] Model OrderContract berhasil dikompilasi dengan konfigurasi strict=True dan extra=forbid.")

[OK] Model OrderContract berhasil dikompilasi dengan konfigurasi strict=True dan extra=forbid.


---
## Percobaan 4: Validasi Satu Record Valid
Menguji record transaksi yang memenuhi semua kriteria kontrak data.

In [4]:
valid_order = {
    "order_id": "ORD-001",
    "customer_id": 101,
    "amount": 250000.0,
    "status": "PAID",
    "transaction_date": datetime(2026, 9, 20, 8, 0),
    "payment_date": datetime(2026, 9, 20, 8, 15),
}

try:
    order = OrderContract.model_validate(valid_order)
    print("STATUS: VALID")
    print("Model Output:", order)
except ValidationError as exc:
    print("STATUS: INVALID")
    print(exc)

STATUS: VALID
Model Output: order_id='ORD-001' customer_id=101 amount=250000.0 status='PAID' transaction_date=datetime.datetime(2026, 9, 20, 8, 0) payment_date=datetime.datetime(2026, 9, 20, 8, 15)


---
## Percobaan 5: Menguji Data Invalid (Constraint Nilai Negatif)
Mengubah `customer_id` menjadi bilangan negatif (`-101`) untuk menguji assertion `gt=0`.

In [5]:
invalid_order_5 = {
    "order_id": "ORD-001",
    "customer_id": -101,  # Melanggar customer_id > 0
    "amount": 250000.0,
    "status": "PAID",
    "transaction_date": datetime(2026, 9, 20, 8, 0),
    "payment_date": datetime(2026, 9, 20, 8, 15),
}

try:
    OrderContract.model_validate(invalid_order_5)
    print("STATUS: VALID")
except ValidationError as exc:
    print("STATUS: INVALID")
    print(exc)

STATUS: INVALID
1 validation error for OrderContract
customer_id
  Input should be greater than 0 [type=greater_than, input_value=-101, input_type=int]


---
## Percobaan 6: Menguji Custom Business Rule
Menguji transaksi dengan status `PAID` tetapi `payment_date` bernilai `None`.

In [6]:
invalid_order_6 = {
    "order_id": "ORD-999",
    "customer_id": 999,
    "amount": 100000.0,
    "status": "PAID",
    "transaction_date": datetime(2026, 9, 20, 10, 0),
    "payment_date": None,  # Melanggar business rule
}

try:
    OrderContract.model_validate(invalid_order_6)
    print("STATUS: VALID")
except ValidationError as exc:
    print("STATUS: INVALID")
    print(exc)

STATUS: INVALID
1 validation error for OrderContract
  Value error, payment_date wajib diisi jika status=PAID [type=value_error, input_value={'order_id': 'ORD-999', '...e, 'payment_date': None}, input_type=dict]


---
## Percobaan 7: Menguji Extra Field (`extra='forbid'`)
Menyisipkan kolom tak terduga (`unexpected_field: 'ABC'`) untuk membuktikan perlindungan terhadap kebocoran skema (*schema drift*).

In [7]:
invalid_order_7 = {
    "order_id": "ORD-888",
    "customer_id": 888,
    "amount": 100000.0,
    "status": "PENDING",
    "transaction_date": datetime(2026, 9, 20, 10, 0),
    "payment_date": None,
    "unexpected_field": "ABC",  # Kolom liar
}

try:
    OrderContract.model_validate(invalid_order_7)
    print("STATUS: VALID")
except ValidationError as exc:
    print("STATUS: INVALID")
    print(exc)

STATUS: INVALID
1 validation error for OrderContract
unexpected_field
  Extra inputs are not permitted [type=extra_forbidden, input_value='ABC', input_type=str]


---
## Percobaan 8 & 9: Validasi Batch Dataset (Valid vs Invalid)
Mengiterasi seluruh baris dataset batch dan mengumpulkan laporan baris yang mengalami galat.

In [8]:
import csv

def parse_datetime(val):
    return datetime.fromisoformat(val) if val else None

def load_orders(path):
    with open(path, newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        rows = []
        for r in reader:
            row_dict = {
                "order_id": r["order_id"],
                "customer_id": int(r["customer_id"]) if r.get("customer_id") and r["customer_id"].lstrip("-").isdigit() else r.get("customer_id"),
                "amount": float(r["amount"]) if r.get("amount") and r["amount"].replace(".", "", 1).lstrip("-").isdigit() else r.get("amount"),
                "status": r["status"],
                "transaction_date": parse_datetime(r.get("transaction_date")),
                "payment_date": parse_datetime(r.get("payment_date")),
            }
            for k, v in r.items():
                if k not in row_dict:
                    row_dict[k] = v
            rows.append(row_dict)
        return rows

def validate_batch(rows):
    valid_orders = []
    errors = []
    for idx, row in enumerate(rows, start=2):
        try:
            order = OrderContract.model_validate(row)
            valid_orders.append(order)
        except ValidationError as exc:
            errors.append({"row": idx, "order_id": row.get("order_id"), "errors": exc.errors()})
    return valid_orders, errors

v_orders, v_err = validate_batch(load_orders('data/orders_valid.csv'))
inv_orders, inv_err = validate_batch(load_orders('data/orders_invalid.csv'))

print("=== HASIL VALIDASI BATCH: orders_valid.csv ===")
print(f"Total Record : {len(v_orders) + len(v_err)}")
print(f"Lolos Valid  : {len(v_orders)}")
print(f"Gagal Cacat  : {len(v_err)}")

print("\n=== HASIL VALIDASI BATCH: orders_invalid.csv ===")
print(f"Total Record : {len(inv_orders) + len(inv_err)}")
print(f"Lolos Valid  : {len(inv_orders)}")
print(f"Gagal Cacat  : {len(inv_err)}")

print("\nDetail Pelanggaran pada Batch Invalid:")
for e in inv_err:
    for sub in e['errors']:
        print(f"Baris {e['row']} ({e['order_id']}): {'.'.join(str(x) for x in sub['loc'])} -> {sub['msg']} ({sub['type']})")

=== HASIL VALIDASI BATCH: orders_valid.csv ===
Total Record : 4
Lolos Valid  : 4
Gagal Cacat  : 0

=== HASIL VALIDASI BATCH: orders_invalid.csv ===
Total Record : 5
Lolos Valid  : 1
Gagal Cacat  : 4

Detail Pelanggaran pada Batch Invalid:
Baris 3 (ORD-102): customer_id -> Input should be greater than 0 (greater_than)
Baris 4 (ORD-103): amount -> Input should be greater than or equal to 0 (greater_than_equal)
Baris 5 (ORD-104): status -> Input should be 'PAID', 'PENDING' or 'CANCELLED' (literal_error)
Baris 6 (ORD-105):  -> Value error, payment_date wajib diisi jika status=PAID (value_error)


---
## Percobaan 10, 11, & 12: Menerapkan Quality Gate Fail-Fast & Simulasi Data Warehouse
Pipeline secara tegas menolak memuat data ke file `data/warehouse_orders.csv` apabila terdapat error pada batch.

In [9]:
def save_to_warehouse(orders, output_path='data/warehouse_orders.csv'):
    with open(output_path, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['order_id', 'customer_id', 'amount', 'status', 'transaction_date', 'payment_date'])
        for o in orders:
            w.writerow([o.order_id, o.customer_id, o.amount, o.status, o.transaction_date.isoformat(), o.payment_date.isoformat() if o.payment_date else ''])

def ingest_batch_pipeline(csv_path):
    rows = load_orders(csv_path)
    valid_orders, errors = validate_batch(rows)
    if errors:
        print(f"-> Evaluasi Quality Gate: FAIL-FAST TRIGGERED! Ditemukan {len(errors)} data cacat.")
        print("-> REJECT: Seluruh batch ditolak. Pemuatan ke Data Warehouse dibatalkan.")
        return False
    print(f"-> Evaluasi Quality Gate: BATCH VALID ({len(valid_orders)} record lolos)")
    save_to_warehouse(valid_orders)
    print(f"-> SUCCESS: Data Warehouse berhasil diperbarui dengan {len(valid_orders)} transaksi baru.")
    return True

print(">>> Menjalankan Pipeline untuk Batch 1 (orders_valid.csv):")
ingest_batch_pipeline('data/orders_valid.csv')

print("\n>>> Menjalankan Pipeline untuk Batch 2 (orders_invalid.csv):")
ingest_batch_pipeline('data/orders_invalid.csv')

print("\n=== ISI AKHIR DATA WAREHOUSE (data/warehouse_orders.csv) ===")
df_warehouse = pd.read_csv('data/warehouse_orders.csv')
print(df_warehouse)
print("\n[PEMBUKTIAN FAIL-FAST BERHASIL]: Data Warehouse tetap bersih dari anomali data cacat.")

>>> Menjalankan Pipeline untuk Batch 1 (orders_valid.csv):
-> Evaluasi Quality Gate: BATCH VALID (4 record lolos)
-> SUCCESS: Data Warehouse berhasil diperbarui dengan 4 transaksi baru.

>>> Menjalankan Pipeline untuk Batch 2 (orders_invalid.csv):
-> Evaluasi Quality Gate: FAIL-FAST TRIGGERED! Ditemukan 4 data cacat.
-> REJECT: Seluruh batch ditolak. Pemuatan ke Data Warehouse dibatalkan.

=== ISI AKHIR DATA WAREHOUSE (data/warehouse_orders.csv) ===
  order_id  customer_id    amount     status     transaction_date         payment_date
0  ORD-001          101  250000.0       PAID  2026-09-20T08:00:00  2026-09-20T08:15:00
1  ORD-002          102  125000.0    PENDING  2026-09-20T08:05:00                  NaN
2  ORD-003          103   75000.0  CANCELLED  2026-09-20T08:10:00                  NaN
3  ORD-004          104  500000.0       PAID  2026-09-20T08:20:00  2026-09-20T08:30:00

[PEMBUKTIAN FAIL-FAST BERHASIL]: Data Warehouse tetap bersih dari anomali data cacat.


---
## Percobaan 13: Dataset-Level Validation dengan Great Expectations (GX Core 1.x)
Melengkapi validasi baris dengan Expectation Suite berbasis dataset tabular, menguji integritas kolom dan distribusi nilai secara deklaratif.

In [10]:
import great_expectations.expectations as gxe

context = gx.get_context(mode='ephemeral')
ds = context.data_sources.add_pandas('gx_source')
asset = ds.add_dataframe_asset('gx_asset')
batch_def = asset.add_batch_definition_whole_dataframe('gx_batch')

suite = gx.ExpectationSuite(name='order_suite')
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='order_id'))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='customer_id', min_value=1))
suite.add_expectation(gxe.ExpectColumnValuesToBeBetween(column='amount', min_value=0.0))
suite.add_expectation(gxe.ExpectColumnValuesToBeInSet(column='status', value_set=['PAID', 'PENDING', 'CANCELLED']))
suite.add_expectation(gxe.ExpectColumnValuesToNotBeNull(column='transaction_date'))
context.suites.add(suite)

val_def = gx.ValidationDefinition(name='order_val_def', data=batch_def, suite=suite)
context.validation_definitions.add(val_def)

res_valid = val_def.run(batch_parameters={'dataframe': df_valid})
res_invalid = val_def.run(batch_parameters={'dataframe': df_invalid})

print("=== HASIL EVALUASI GX PADA orders_valid.csv ===")
print(f"Status Keseluruhan : {'PASS' if res_valid.success else 'FAIL'} ({res_valid.success})")
for r in res_valid.results:
    print(f" - [{'PASSED' if r.success else 'FAILED'}] {r.expectation_config.type} pada kolom '{r.expectation_config.kwargs.get('column')}'")

print("\n=== HASIL EVALUASI GX PADA orders_invalid.csv ===")
print(f"Status Keseluruhan : {'PASS' if res_invalid.success else 'FAIL'} ({res_invalid.success})")
for r in res_invalid.results:
    exp_name = r.expectation_config.type
    col = r.expectation_config.kwargs.get('column')
    st = 'PASSED' if r.success else 'FAILED'
    detail = f" ({r.result.get('unexpected_count')} anomali: {r.result.get('partial_unexpected_list')})" if not r.success else ""
    print(f" - [{st}] {exp_name} pada kolom '{col}'{detail}")

=== HASIL EVALUASI GX PADA orders_valid.csv ===
Status Keseluruhan : PASS (True)
 - [PASSED] expect_column_values_to_not_be_null pada kolom 'order_id'
 - [PASSED] expect_column_values_to_be_between pada kolom 'customer_id'
 - [PASSED] expect_column_values_to_be_between pada kolom 'amount'
 - [PASSED] expect_column_values_to_be_in_set pada kolom 'status'
 - [PASSED] expect_column_values_to_not_be_null pada kolom 'transaction_date'

=== HASIL EVALUASI GX PADA orders_invalid.csv ===
Status Keseluruhan : FAIL (False)
 - [PASSED] expect_column_values_to_not_be_null pada kolom 'order_id'
 - [FAILED] expect_column_values_to_be_between pada kolom 'customer_id' (1 anomali: [-202])
 - [FAILED] expect_column_values_to_be_between pada kolom 'amount' (1 anomali: [-75000.0])
 - [FAILED] expect_column_values_to_be_in_set pada kolom 'status' (1 anomali: ['UNKNOWN'])
 - [PASSED] expect_column_values_to_not_be_null pada kolom 'transaction_date'


---
## Kesimpulan & Komparasi: Pydantic vs Great Expectations

| Aspek Perbandingan | Pydantic v2 | Great Expectations (GX Core 1.x) |
|---|---|---|
| **Fokus Operasi** | *Row-level validation* & *Object mapping* pada batas data masuk (*ingestion boundary*). | *Dataset-level validation*, *batch profiling*, dan *data documentation*. |
| **Peran Arsitektur** | Memvalidasi integritas tipe data, strictness skema, dan *business logic* kompleks antar-field sebelum data diproses. | Menguji karakteristik agregat dataset (rentang nilai, keunikan, *null rate*, distribusi). |
| **Kelebihan Utama** | Sangat cepat, terintegrasi mulus dengan ekosistem aplikasi Python/FastAPI/Pydantic, mendukung custom conditional validator. | Menghasilkan laporan kualitas visual (*Data Docs*), terintegrasi baik dengan engine SQL & Spark, deklaratif berbasis metadata. |
| **Penerapan Pipeline** | Gerbang pertama (*Front Quality Gate*) saat menerima event/batch data dari Data Producer. | Gerbang kedua (*Downstream Quality Gate*) untuk audit integritas tabel analitik di Data Warehouse/Data Lake. |